# Calling DMRs with cytozip

This notebook walks through the DMR-calling APIs in cytozip:

| API | Use case |
|---|---|
| `czip.call_dmr` | **CG** (CpG) DMRs between two groups of `.cz` files (single-cell or pseudobulk) |
| `czip.call_dmr_ch` | **non-CG** (CH / CA / CT) DMRs with bin aggregation, global mCH normalization, and a log2-fold-change filter |
| `czip.call_dmr_one_vs_rest` | Batch one-vs-rest over a folder of pseudobulk `.cz` files; optionally **stratified** within each class given a 2-column TSV |
| `czip.merge_dmr_results` | Stitch per-sample TSVs into a single long-format table with derived metrics (`delta_meth` / `delta_rate`, `direction`, `log2fc`, `length`, `region_id`) |

All DMR callers share the same OpenMP-parallel Cython RMS permutation kernel (Schultz et al. 2015 / ALLCools).

## Pseudobulk vs single-cell input

`group_a` / `group_b` accept **either** single-cell `.cz` files **or** pseudobulk `.cz` files. **Pseudobulk is strongly recommended** for cell-type-vs-cell-type comparisons:

* Per-cell coverage is sparse (CG cov is mostly 0/1; CH is essentially zero), so per-site permutation tests on single cells have limited power.
* Pooling cells of the same cluster gives stable per-site coverage and matches ALLCools' / Methylpy's "one ALLC = one sample" convention.
* Runtime scales linearly with the number of input files, so going from N cells to K clusters is a large speed-up.

Recommended workflow: build **multiple replicate pseudobulks per cluster** with `czip merge_cz` (split by donor / batch, or random splits), then pass those replicate pseudobulks as `group_a` / `group_b`.

In [1]:
import os
os.chdir(os.path.expanduser("~/Projects/Github/cytozip/cytozip_example_data"))
import glob
import pandas as pd
import cytozip as czip

## 1. CG DMR — `call_dmr`

`call_dmr` re-implements ALLCools' permutation root-mean-square (RMS) test on the `.cz` format. The hot loop lives in the Cython extension `cytozip.dmr_accel` (OpenMP `prange`, dynamic schedule) and the outer chunk loop is split into `processes × threads ≈ jobs`.

Algorithm:
1. For each reference chunk, load each cell's `mc`/`cov` columns into an `(n_cells, n_sites)` int32 matrix.
2. Keep sites with at least `min_samples_per_group` cells passing `cov >= min_cov` in both groups.
3. Run the RMS permutation test per site (early stop at `min_pvalue`, downsampling caps `max_row_count`/`max_total_count` matching ALLCools defaults).
4. Sites with `p < p_value_cutoff` and `|mean(frac_A) - mean(frac_B)| > frac_delta_cutoff` are DMS.
5. Adjacent same-sign DMS within `max_dist` bp are merged into DMRs (`min_dms` per DMR).

In [2]:
cells = sorted(glob.glob("output/cz/FC_*.cz"))
group_a, group_b = cells[:5], cells[5:]

czip.call_dmr(
    group_a=group_a, group_b=group_b,
    reference="output/mm10_with_chrL.allc.cz",
    output="/tmp/cz_dmr_demo.tsv",
    dms_output="/tmp/cz_dms_demo.tsv",
    chroms=["chrM", "chrL"],   # restrict to small chroms for the demo
    min_cov=1, min_samples_per_group=1,
    p_value_cutoff=0.05, frac_delta_cutoff=0.2,
    n_permute=2000, jobs=4,
)

pd.read_csv("/tmp/cz_dmr_demo.tsv", sep="\t").head()

2026-04-27 14:23:24.709 | INFO     | cytozip.dmr:call_dmr:509 - call_dmr: 5 cells in A vs 4 cells in B over 2 chunks (jobs=4 = 2 proc x 2 threads)


2026-04-27 14:23:30.300 | INFO     | cytozip.dmr:call_dmr:587 - DMS table (841 sites) written to /tmp/cz_dms_demo.tsv


2026-04-27 14:23:30.307 | INFO     | cytozip.dmr:call_dmr:594 - call_dmr: 841 DMS -> 181 DMRs written to /tmp/cz_dmr_demo.tsv


,chrom,start,end,n_dms,p_min,frac_delta_mean,state
0,chrL,407,596,8,0.009500,0.255072,1
1,chrL,972,973,1,0.042084,0.346425,1
2,chrL,1282,1283,1,0.026382,0.235873,1
3,chrL,1336,1475,15,0.001500,-0.325388,-1
4,chrL,1616,1646,3,0.036585,0.361111,1


Output columns (BED-style, 0-based half-open):

| column | meaning |
|---|---|
| `chrom`, `start`, `end` | DMR coordinates |
| `n_dms` | number of significant sites in the DMR |
| `p_min` | smallest per-site p-value in the DMR |
| `frac_delta_mean` | mean `frac_A − frac_B` over constituent DMS |
| `state` | `+1` = hyper in A, `-1` = hypo in A, `0` = mixed |

## 2. CH DMR — `call_dmr_ch`

CH methylation has very low fractions (~0.5–5% in neurons, much less elsewhere) and large global-rate differences between cell types, so the CG-tuned `call_dmr` is unsuitable. `call_dmr_ch` adapts the same RMS kernel:

1. **Bin aggregation** — pool per-cell mc/cov into fixed-size bins (default 5 kb) so the test has signal.
2. **Global-rate normalization** (default ON) — pre-compute each cell's global mCH rate and rescale its mc counts to a common median target. This removes the dominant cell-type-level global shift that would otherwise saturate the test.
3. **Triple filter** — DMS bins must satisfy `p < p_value_cutoff` AND `|log2(rate_A / rate_B)| ≥ log2fc_cutoff` AND `|rate_A − rate_B| ≥ abs_delta_cutoff`.
4. **Sub-context restriction** — pass `index=mm10.CAN.cz` to analyze mCA only (recommended in mammalian neurons).
5. Wider merge distance (`max_dist=2000`) and `min_dms=2` since CH signal is broader.

For CH, **pseudobulk input is even more strongly recommended** than for CG: per-cell per-site CH coverage is essentially zero, so even with binning the signal in single-cell input is dominated by sampling noise.

In [3]:
czip.call_dmr_ch(
    group_a=group_a, group_b=group_b,
    reference="output/mm10_with_chrL.allc.cz",
    output="/tmp/cz_dmr_ch_demo.tsv",
    dms_output="/tmp/cz_dms_ch_demo.tsv",
    bin_size=1000,                  # smaller bins for the tiny demo dataset
    chroms=["chrM", "chrL"],
    p_value_cutoff=0.05,
    log2fc_cutoff=0.3,
    abs_delta_cutoff=0.01,
    min_cov=1, min_samples_per_group=2,
    n_permute=3000, jobs=4,
)

pd.read_csv("/tmp/cz_dms_ch_demo.tsv", sep="\t").head()

2026-04-27 14:23:30.345 | INFO     | cytozip.dmr:call_dmr_ch:990 - call_dmr_ch[CHN]: 5 cells in A vs 4 cells in B over 2 chunks, bin_size=1000 (jobs=4 = 2 proc x 2 threads)


2026-04-27 14:23:30.345 | INFO     | cytozip.dmr:call_dmr_ch:1000 - call_dmr_ch: scanning group A for per-cell global rates


2026-04-27 14:23:30.580 | INFO     | cytozip.dmr:call_dmr_ch:1009 - call_dmr_ch: scanning group B for per-cell global rates


2026-04-27 14:23:30.821 | INFO     | cytozip.dmr:call_dmr_ch:1030 - call_dmr_ch: normalized to global rate target=0.6130; A scale range [0.93,1.01], B scale range [1.00,1.05]


2026-04-27 14:23:31.508 | INFO     | cytozip.dmr:call_dmr_ch:1118 - DMS bins (12) written to /tmp/cz_dms_ch_demo.tsv


2026-04-27 14:23:31.512 | INFO     | cytozip.dmr:call_dmr_ch:1143 - call_dmr_ch[CHN]: 12 DMS bins -> 2 DMRs written to /tmp/cz_dmr_ch_demo.tsv


,chrom,pos,p,delta,rate_a,rate_b
0,chrM,1,0.000333,0.181566,0.727273,0.545706
1,chrM,1001,0.000333,0.259249,0.782665,0.523416
2,chrM,2001,0.000333,0.321738,0.689085,0.367347
3,chrM,3001,0.000333,0.258977,0.777049,0.518072
4,chrM,4001,0.000333,0.383304,0.889032,0.505728


On real single-cell data, `call_dmr_ch` is typically **5–50× faster than `call_dmr`** because bin aggregation collapses tens of thousands of per-site tests into a few thousand per-bin tests, even with the optional global-rate pre-scan.

## 3. Batch one-vs-rest — `call_dmr_one_vs_rest`

Scans a directory of pseudobulk `.cz` files, runs **each sample vs. all the others** in one go, and (after the last comparison) auto-merges per-sample TSVs into a single long-format table at `<outdir>/merged_dmr.tsv` with derived metrics.

* `class_table=None` → **global** one-vs-rest (output: `outdir/<sname>.dmr.tsv`).
* `class_table=...` (TSV path / `dict` / `DataFrame` with two columns `sname<TAB>class`) → **stratified**: each comparison is restricted to other members of the same class (output: `outdir/<class>/<sname>.dmr.tsv`).
* `method='cg' | 'ch'` switches between `call_dmr` and `call_dmr_ch`; extra knobs (e.g. `bin_size`, `frac_delta_cutoff`, `log2fc_cutoff`, `n_permute`, …) flow through via `**dmr_kwargs`.
* `overwrite=False` skips comparisons whose output TSV already exists (resume-friendly).
* For CH, the per-pseudobulk global mCH rates are pre-scanned **once** at the wrapper level and reused across all comparisons via `global_a` / `global_b`, instead of re-scanning on every call.

In [4]:
# Treat each cell .cz as a (toy) pseudobulk and stage them in a folder.
import shutil
tmpd = "/tmp/cz_pb_demo"
shutil.rmtree(tmpd, ignore_errors=True); os.makedirs(tmpd)
for p in cells:
    os.symlink(os.path.abspath(p), os.path.join(tmpd, os.path.basename(p)))

outdir = "/tmp/cz_dmr_ovr_global"
shutil.rmtree(outdir, ignore_errors=True)

czip.call_dmr_one_vs_rest(
    indir=tmpd,
    reference="output/mm10_with_chrL.allc.cz",
    outdir=outdir,
    ext=".cz",
    method="cg",
    p_value_cutoff=0.05, frac_delta_cutoff=0.2,
    min_cov=1, min_samples_per_group=2,
    n_permute=2000,
    chroms=["chrM", "chrL"],
    jobs=4,
)

# Per-sample DMR TSVs:
print(sorted(os.listdir(outdir))[:5], "...")
# Auto-merged table with metric columns:
merged = pd.read_csv(os.path.join(outdir, "merged_dmr.tsv"), sep="\t")
print("merged shape:", merged.shape)
merged.head()

2026-04-27 14:23:31.532 | INFO     | cytozip.dmr:call_dmr_one_vs_rest:1448 - [1/9] global: FC_E17b_3C_5-5-I24-A21 vs rest (8 samples)


2026-04-27 14:23:31.534 | INFO     | cytozip.dmr:call_dmr:509 - call_dmr: 1 cells in FC_E17b_3C_5-5-I24-A21 vs 8 cells in rest over 2 chunks (jobs=4 = 2 proc x 2 threads)


2026-04-27 14:23:31.811 | INFO     | cytozip.dmr:call_dmr:594 - call_dmr: 0 DMS -> 0 DMRs written to /tmp/cz_dmr_ovr_global/FC_E17b_3C_5-5-I24-A21.dmr.tsv


2026-04-27 14:23:31.812 | INFO     | cytozip.dmr:call_dmr_one_vs_rest:1448 - [2/9] global: FC_M_E15a_3C_1-1-I5-B1 vs rest (8 samples)


2026-04-27 14:23:31.813 | INFO     | cytozip.dmr:call_dmr:509 - call_dmr: 1 cells in FC_M_E15a_3C_1-1-I5-B1 vs 8 cells in rest over 2 chunks (jobs=4 = 2 proc x 2 threads)


...

2026-04-27 14:23:33.558 | INFO     | cytozip.dmr:call_dmr:509 - call_dmr: 1 cells in FC_P28a_3C_2-1-E5-N14 vs 8 cells in rest over 2 chunks (jobs=4 = 2 proc x 2 threads)


2026-04-27 14:23:33.799 | INFO     | cytozip.dmr:call_dmr:594 - call_dmr: 0 DMS -> 0 DMRs written to /tmp/cz_dmr_ovr_global/FC_P28a_3C_2-1-E5-N14.dmr.tsv


2026-04-27 14:23:33.811 | INFO     | cytozip.dmr:merge_dmr_results:1641 - merge_dmr_results: 0 DMRs across 0 samples written to /tmp/cz_dmr_ovr_global/merged_dmr.tsv


['FC_E17b_3C_5-5-I24-A21.dmr.tsv', 'FC_M_E15a_3C_1-1-I5-B1.dmr.tsv', 'FC_M_P12b_3C_2-5-M17-N10.dmr.tsv', 'FC_M_P3b_3C_6-6-J3-P24.dmr.tsv', 'FC_M_P6a_3C_7-3-K21-P5.dmr.tsv'] ...
merged shape: (0, 9)


,chrom,start,end,n_dms,p_min,frac_delta_mean,state,sname,class


### Stratified one-vs-rest (within each class)

Provide a 2-column TSV (`sname<TAB>class`, no header) — only members of the same class are compared:

In [5]:
# Build a toy class table: split the 9 cells into two classes
class_tsv = "/tmp/cz_class_table.tsv"
with open(class_tsv, "w") as f:
    for i, p in enumerate(cells):
        sname = os.path.basename(p)[:-len(".cz")]
        f.write(f"{sname}\tcls{1 if i < 4 else 2}\n")
pd.read_csv(class_tsv, sep="\t", header=None, names=["sname","class"]).head()

,sname,class
0,FC_E17b_3C_5-5-I24-A21,cls1
1,FC_M_E15a_3C_1-1-I5-B1,cls1
2,FC_M_P12b_3C_2-5-M17-N10,cls1
3,FC_M_P3b_3C_6-6-J3-P24,cls1
4,FC_M_P6a_3C_7-3-K21-P5,cls2


In [6]:
outdir2 = "/tmp/cz_dmr_ovr_strat"
shutil.rmtree(outdir2, ignore_errors=True)

czip.call_dmr_one_vs_rest(
    indir=tmpd,
    reference="output/mm10_with_chrL.allc.cz",
    outdir=outdir2,
    ext=".cz",
    method="cg",
    class_table=class_tsv,
    p_value_cutoff=0.05, frac_delta_cutoff=0.2,
    min_cov=1, min_samples_per_group=2,
    n_permute=2000,
    chroms=["chrM", "chrL"],
    jobs=4,
)

for d in sorted(os.listdir(outdir2)):
    sub = os.path.join(outdir2, d)
    if os.path.isdir(sub):
        print(d, "->", sorted(os.listdir(sub)))

2026-04-27 14:23:33.831 | INFO     | cytozip.dmr:call_dmr_one_vs_rest:1448 - [1/9] cls1: FC_E17b_3C_5-5-I24-A21 vs rest (3 samples)


2026-04-27 14:23:33.832 | INFO     | cytozip.dmr:call_dmr:509 - call_dmr: 1 cells in FC_E17b_3C_5-5-I24-A21 vs 3 cells in rest over 2 chunks (jobs=4 = 2 proc x 2 threads)


2026-04-27 14:23:34.074 | INFO     | cytozip.dmr:call_dmr:594 - call_dmr: 0 DMS -> 0 DMRs written to /tmp/cz_dmr_ovr_strat/cls1/FC_E17b_3C_5-5-I24-A21.dmr.tsv


2026-04-27 14:23:34.074 | INFO     | cytozip.dmr:call_dmr_one_vs_rest:1448 - [2/9] cls1: FC_M_E15a_3C_1-1-I5-B1 vs rest (3 samples)


2026-04-27 14:23:34.075 | INFO     | cytozip.dmr:call_dmr:509 - call_dmr: 1 cells in FC_M_E15a_3C_1-1-I5-B1 vs 3 cells in rest over 2 chunks (jobs=4 = 2 proc x 2 threads)


...

2026-04-27 14:23:35.774 | INFO     | cytozip.dmr:call_dmr_one_vs_rest:1448 - [9/9] cls2: FC_P28a_3C_2-1-E5-N14 vs rest (4 samples)


2026-04-27 14:23:35.775 | INFO     | cytozip.dmr:call_dmr:509 - call_dmr: 1 cells in FC_P28a_3C_2-1-E5-N14 vs 4 cells in rest over 2 chunks (jobs=4 = 2 proc x 2 threads)


2026-04-27 14:23:36.004 | INFO     | cytozip.dmr:call_dmr:594 - call_dmr: 0 DMS -> 0 DMRs written to /tmp/cz_dmr_ovr_strat/cls2/FC_P28a_3C_2-1-E5-N14.dmr.tsv


2026-04-27 14:23:36.014 | INFO     | cytozip.dmr:merge_dmr_results:1641 - merge_dmr_results: 0 DMRs across 0 samples written to /tmp/cz_dmr_ovr_strat/merged_dmr.tsv


cls1 -> ['FC_E17b_3C_5-5-I24-A21.dmr.tsv', 'FC_M_E15a_3C_1-1-I5-B1.dmr.tsv', 'FC_M_P12b_3C_2-5-M17-N10.dmr.tsv', 'FC_M_P3b_3C_6-6-J3-P24.dmr.tsv']
cls2 -> ['FC_M_P6a_3C_7-3-K21-P5.dmr.tsv', 'FC_M_P9B_3C_6-2-F6-O4.dmr.tsv', 'FC_P0b_3C_5-1-I24-J14.dmr.tsv', 'FC_P13a_3C_7-1-A11-O1.dmr.tsv', 'FC_P28a_3C_2-1-E5-N14.dmr.tsv']


### CH batch one-vs-rest with shared global-rate pre-scan

Switch to `method='ch'` and pass CH-specific knobs through `**dmr_kwargs`. The wrapper logs `pre-scanning global mCH rates for N pseudobulks` once at the start, then reuses those rates for every comparison.

In [7]:
outdir3 = "/tmp/cz_dmr_ovr_ch"
shutil.rmtree(outdir3, ignore_errors=True)

czip.call_dmr_one_vs_rest(
    indir=tmpd,
    reference="output/mm10_with_chrL.allc.cz",
    outdir=outdir3,
    ext=".cz",
    method="ch",
    bin_size=1000,
    p_value_cutoff=0.05,
    log2fc_cutoff=0.3,
    abs_delta_cutoff=0.01,
    min_cov=1, min_samples_per_group=2,
    n_permute=2000,
    chroms=["chrM", "chrL"],
    jobs=4,
)

print(sorted(os.listdir(outdir3))[:5], "...")

2026-04-27 14:23:36.020 | INFO     | cytozip.dmr:call_dmr_one_vs_rest:1385 - call_dmr_one_vs_rest[CH]: pre-scanning global mCH rates for 9 pseudobulks


2026-04-27 14:23:36.264 | INFO     | cytozip.dmr:call_dmr_one_vs_rest:1391 -   global rate range [0.5811, 0.6593], median=0.6130


2026-04-27 14:23:36.264 | INFO     | cytozip.dmr:call_dmr_one_vs_rest:1448 - [1/9] global: FC_E17b_3C_5-5-I24-A21 vs rest (8 samples)


2026-04-27 14:23:36.265 | INFO     | cytozip.dmr:call_dmr_ch:990 - call_dmr_ch[CHN]: 1 cells in FC_E17b_3C_5-5-I24-A21 vs 8 cells in rest over 2 chunks, bin_size=1000 (jobs=4 = 2 proc x 2 threads)


2026-04-27 14:23:36.266 | INFO     | cytozip.dmr:call_dmr_ch:1030 - call_dmr_ch: normalized to global rate target=0.6130; A scale range [0.93,0.93], B scale range [0.99,1.05]


...

2026-04-27 14:23:38.321 | INFO     | cytozip.dmr:call_dmr_ch:990 - call_dmr_ch[CHN]: 1 cells in FC_P28a_3C_2-1-E5-N14 vs 8 cells in rest over 2 chunks, bin_size=1000 (jobs=4 = 2 proc x 2 threads)


2026-04-27 14:23:38.321 | INFO     | cytozip.dmr:call_dmr_ch:1030 - call_dmr_ch: normalized to global rate target=0.6130; A scale range [1.05,1.05], B scale range [0.93,1.05]


2026-04-27 14:23:38.567 | INFO     | cytozip.dmr:call_dmr_ch:1143 - call_dmr_ch[CHN]: 0 DMS bins -> 0 DMRs written to /tmp/cz_dmr_ovr_ch/FC_P28a_3C_2-1-E5-N14.dmr.tsv


2026-04-27 14:23:38.577 | INFO     | cytozip.dmr:merge_dmr_results:1641 - merge_dmr_results: 0 DMRs across 0 samples written to /tmp/cz_dmr_ovr_ch/merged_dmr.tsv


['FC_E17b_3C_5-5-I24-A21.dmr.tsv', 'FC_M_E15a_3C_1-1-I5-B1.dmr.tsv', 'FC_M_P12b_3C_2-5-M17-N10.dmr.tsv', 'FC_M_P3b_3C_6-6-J3-P24.dmr.tsv', 'FC_M_P6a_3C_7-3-K21-P5.dmr.tsv'] ...


## 4. Merging results — `merge_dmr_results`

`call_dmr_one_vs_rest` calls this automatically at the end, but you can also re-run it manually (e.g. with a different `class_table` or to refresh metric columns).

Always-present columns: `chrom, start, end, n_dms, p_min, frac_delta_mean, state, sname, class, length, region_id`.

When `add_metrics=True` (default):

* CG → adds `delta_meth` (= `frac_delta_mean`) and `direction` (`'hypo'` / `'hyper'`).
* CH → adds `delta_rate` and `direction`; if `rate_a` / `rate_b` columns are present (e.g. when merging the per-bin DMS files via `dms_output_dir`) it also computes `log2fc = log2(rate_a / rate_b)`.

In [8]:
# Synthetic example to illustrate the metric columns when there are real hits.
demo = "/tmp/cz_merge_demo"
shutil.rmtree(demo, ignore_errors=True)
os.makedirs(os.path.join(demo, "cls1"))
os.makedirs(os.path.join(demo, "cls2"))

pd.DataFrame({
    "chrom": ["chr1", "chr1"],
    "start": [100, 500],
    "end":   [300, 700],
    "n_dms": [3, 5],
    "p_min": [1e-4, 2e-3],
    "frac_delta_mean": [-0.4, 0.3],
    "state": [-1, 1],
}).to_csv(f"{demo}/cls1/sampA.dmr.tsv", sep="\t", index=False)

pd.DataFrame({
    "chrom": ["chr2"],
    "start": [1000], "end": [2000],
    "n_dms": [2], "p_min": [5e-5],
    "frac_delta_mean": [0.55], "state": [1],
}).to_csv(f"{demo}/cls2/sampB.dmr.tsv", sep="\t", index=False)

merged = czip.merge_dmr_results(demo, method="cg",
                                output=f"{demo}/merged.tsv")
merged

2026-04-27 14:23:38.594 | INFO     | cytozip.dmr:merge_dmr_results:1641 - merge_dmr_results: 3 DMRs across 2 samples written to /tmp/cz_merge_demo/merged.tsv


,chrom,start,end,n_dms,p_min,frac_delta_mean,state,sname,class,length,region_id,direction,delta_meth,q_min
0,chr1,100,300,3,0.00010,-0.40,-1,sampA,cls1,200,chr1:100-300,hypo,-0.40,0.00020
1,chr1,500,700,5,0.00200,0.30,1,sampA,cls1,200,chr1:500-700,hyper,0.30,0.00200
2,chr2,1000,2000,2,0.00005,0.55,1,sampB,cls2,1000,chr2:1000-2000,hyper,0.55,0.00005


## 5. CLI equivalents

All four functions are also available as `czip` subcommands:

```bash
# Two-group CG
czip call_dmr -a a.txt -b b.txt -r mm10.allc.cz \
    -s mm10.CGN.cz -O dmr.tsv -j 16

# Two-group CH
czip call_dmr_ch -a a.txt -b b.txt -r mm10.allc.cz \
    -s mm10.CAN.cz -O ch_dmr.tsv --bin_size 5000 -j 16

# Batch one-vs-rest, optionally stratified, auto-merge:
czip call_dmr_one_vs_rest \
    -d pseudobulk/Subclass/cz \
    -r mm10.allc.cz \
    -O DMR/Subclass \
    --ext .CGN.cz --method cg \
    -c Subclass2CellClass.tsv \
    -j 32
```

`-a` / `-b` accept comma-separated paths or a text file with one path per line.

## 6. Per-cell methylation over DMRs — `cz_to_anndata`

Once you have a merged DMR table (or any BED) you can directly aggregate **single-cell** `.cz` files over those intervals using `czip.cz_to_anndata` — no extra wrapper needed.

* `features=` accepts a BED/BED.gz path, a `DataFrame` whose first 3 columns are `chrom/start/end` (4th column → `var_names`), or an int (genome-wide bin size). The merged DMR table fits the DataFrame form directly.
* `score='frac'` writes raw `mc/cov` per (cell, DMR) into `.X`; `.layers['mc']` / `.layers['cov']` always carry the raw integer counts so downstream Beta-binomial / posterior models can recompute fractions.
* For mCH DMRs, point `reference=` at the matching CH index (e.g. `mm10.CAN.cz`) — `score='frac'` then yields per-cell mCH rates.
* `jobs>1` runs one cell per worker via `ProcessPoolExecutor`.

In [9]:
# Use the merged CG DMR table from section 3 as a feature set
merged = pd.read_csv(os.path.join(outdir, "merged_dmr.tsv"), sep="\t")
feats = (merged[["chrom", "start", "end", "region_id"]]
         .drop_duplicates(subset="region_id")
         .reset_index(drop=True))
# parse_features uses positional column 0/1/2/3, so rename for safety
feats.columns = [0, 1, 2, 3]
feats.head()

KeyError: "['region_id'] not in index"

In [10]:
adata = czip.cz_to_anndata(
    cz_inputs=cells,                       # list / dir / merged catcz all OK
    features=feats,
    reference="output/mm10_with_chrL.allc.cz",
    score="frac",                          # 'hypo-score' / 'hyper-score' / 'mc' / 'cov' / 'umc'
    exclude_chroms=(),                     # keep chrL/chrM for the demo
    output="/tmp/cz_dmr_per_cell.h5ad",
    jobs=4,
)
print(adata)
print("X dtype:", adata.X.dtype, "layers:", list(adata.layers.keys()))
adata.to_df().head()

NameError: name 'feats' is not defined

Tips:

* The same call works for CH DMRs — substitute `reference="mm10.CAN.cz"` (or whichever sub-context index matches your CH `.cz` files) and `score='frac'` returns per-cell mCH rates over each DMR.
* For binarized hypo/hyper masks, use `score='hypo-score'` (or `'hyper-score'`) with `score_cutoff=0.9`.
* CLI equivalent: convert the merged DMR table to a 4-column BED first, then

  ```bash
  czip cz_to_anndata -i single_cell/cz -f dmrs.bed \
      -r mm10.allc.cz -O DMR_per_cell.h5ad -j 32
  ```